# Project Overview: Real-Time Streaming Data Processing with Databricks

This project demonstrates how to work with real-time streaming data in Databricks using Spark Structured Streaming. The objective is to showcase Databricks’ ability to ingest, process, and store streaming data efficiently for near real-time analytics.

The project focuses on the following key components:

- Simulating real-time sales data streams.
- Processing streaming data and persisting it into Delta tables.
- Running real-time queries on continuously ingested data.

Scenario
A retail organization requires a real-time data processing pipeline to capture live sales transactions from multiple stores. 

The goal is to ingest streaming sales data as it arrives, process it with minimal latency, and store it in Delta tables to support real-time analysis and reporting. This pipeline enables the organization to monitor sales activity, identify trends as they happen, and support timely business decisions.

By implementing this project, you gain hands-on experience with building end-to-end streaming data pipelines in Databricks using Spark Structured Streaming and Delta Lake.

## Step 1: Simulate Real-Time Data Streams

In [0]:
import json
import time
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

In [0]:
root                 = 'dbfs:/FileStore'
streaming_directory  = f'{root}/tables/stream_data'
checkpoint_path      = f'{root}/checkpoints/streaming_query'
delta_table_path     = f'{root}/tables/streaming_table'


In [0]:
dbutils.fs.rm(streaming_directory, recurse=True)
dbutils.fs.rm(checkpoint_path, recurse=True)
dbutils.fs.rm(delta_table_path, recurse=True)

dbutils.fs.mkdirs(streaming_directory)  
dbutils.fs.mkdirs(checkpoint_path)  
dbutils.fs.mkdirs(delta_table_path)

## Step 2: Define the Schema and Load Streaming Data

In [0]:
# mock streaming data
for i in range(10):
    data = {"product":f"Product-{i}", "quantity":i * 2, "timestamp": int(time.time())}
    dbutils.fs.put(f"{streaming_directory}/stream_data_{i}.json", json.dumps(data), overwrite=True)
    time.sleep(2)
print(f'Streaming data generated')

In [0]:
schema = StructType([
    StructField('product', StringType(), True),
    StructField('quantity', IntegerType(), True),
    StructField('timestamp', LongType(), True)
])

## Step 3: Process and Save Streaming Data to a Delta Table

In [0]:
streaming_df = (
    spark.readStream
    .schema(schema)
    .option("maxFilesPerTrigger", 1)
    .json(streaming_directory)
)

query = (
    streaming_df
        .writeStream
        .outputMode('append')
        .option('checkpointLocation', checkpoint_path)
        .trigger(availableNow=True)
        .start(delta_table_path)
)
query.awaitTermination()


## Step 4: Register and Query the Streaming Delta Table

In [0]:
streaming_df = spark.read.format('delta').load(delta_table_path)
streaming_df.display()